In order to make sure all packages are installed properly and data can be loaded, we follow an example from Chollet's book (though, with a different dataset).  

In [1]:
import os, glob, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Rescaling, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import image_dataset_from_directory

Tensorflow was setup incorrectly, so I followed https://yashguptatech.medium.com/tensorflow-setup-on-apple-silicon-mac-m1-m1-pro-m1-max-661d4a6fbb77 to get it to work properly.  The output below should match item 13 on that site.

In [2]:
print(f"TensorFlow has access to the following devices:\n{tf.config.list_physical_devices()}")
print(f"TensorFlow version: {tf.__version__}")

TensorFlow has access to the following devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.16.2


In [3]:
PATH = Path('.').absolute().parent
data_dir = PATH / 'data' / 'fruits-360_dataset_100x100' / 'fruits-360'

IMG_SIZE = (100, 100)
BATCH_SIZE = 32
SEED = 1984

See page 217 in Deep Learning, Chollet, for information about loading an image dataset from a directory in TensorFlow.

In [4]:
train_dataset = image_dataset_from_directory(
    data_dir / 'Training',
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED
)

test_dataset = image_dataset_from_directory(
    data_dir / 'Test',
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED
)

Found 70491 files belonging to 141 classes.


2025-02-18 19:38:13.850711: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2025-02-18 19:38:13.850751: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-02-18 19:38:13.850771: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-02-18 19:38:13.850794: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-02-18 19:38:13.850806: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 23619 files belonging to 141 classes.


In [6]:
for data_batch, labels_batch in train_dataset:
    print('data batch shape:', data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    break

data batch shape: (32, 100, 100, 3)
labels batch shape: (32,)


In [8]:
model = Sequential([
    Input(shape=(100, 100, 3)),
    Rescaling(1./255),
    Conv2D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling2D(pool_size=2),
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling2D(pool_size=2),
    Conv2D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling2D(pool_size=2),
    Conv2D(filters=256, kernel_size=3, activation='relu'),
    MaxPooling2D(pool_size=2),
    Flatten(),
    Dense(141, activation='softmax'),
])

In [10]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 141)            │       577,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 966,093 (3.69 MB)

 Trainable params: 966,093 (3.69 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [16]:
history = model.fit(
    train_dataset,
    epochs=30
)

Epoch 1/30


2025-02-18 19:49:09.048838: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-02-18 19:49:09.054535: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


2203/2203 ━━━━━━━━━━━━━━━━━━━━ 58s 24ms/step - accuracy: 0.7661 - loss: 0.9777
Epoch 2/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 53s 24ms/step - accuracy: 0.9850 - loss: 0.0515
Epoch 3/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9959 - loss: 0.0150
Epoch 4/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 53s 24ms/step - accuracy: 0.9984 - loss: 0.0058
Epoch 5/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9926 - loss: 0.0278
Epoch 6/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9975 - loss: 0.0098
Epoch 7/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 53s 24ms/step - accuracy: 0.9965 - loss: 0.0134
Epoch 8/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9966 - loss: 0.0120
Epoch 9/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 53s 24ms/step - accuracy: 0.9971 - loss: 0.0122
Epoch 10/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9963 - loss: 0.0145
Epoch 11/30
2203/2203 ━━━━━━━━━━━━━━━━━━━━ 54s 25ms/step - accuracy: 1.0000 - loss: 6.8218e-06
Epoch 12/30
220

In [17]:
test_loss, test_acc = model.evaluate(test_dataset)
print(f'Test accuracy: {test_acc}')

  5/739 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9818 - loss: 0.3607   

2025-02-18 20:16:09.077275: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


739/739 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9646 - loss: 0.4081
Test accuracy: 0.9646894335746765
